In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

In [ ]:
def plot_TS(timesteps,values, format=".", start=0, end = None, label = None,X_label='Time',y_label= "Value of EURUSD" , Legend = False , color='r'):
    """
    plots timesteps (a series of points in time) against the values (a series of values across timesteps)

    parameters : timesteps,values, format="." of plot, start=0 of plot, end = None of plot, label = None of the plot
    """
    plt.plot(timesteps[start:end],values,format, label = label,c=color)
    plt.xlabel(X_label)
    plt.ylabel(y_label)

    if Legend:
        plt.legend()
    # if label:
    #     plt.title(label)
    plt.grid(True)

# def denormalize(forecast) :
#     max1 = 999.0
#     min1 = 0.9
#     max2 = 4996.0 
#     min2 = 14.37
#     denormalized_forecast = forecast * (max1 - min1) + min1 
#     denormalized_forecast_2 = forecast * (max2 - min2) + min2 
    
#     return (denormalized_forecast + denormalized_forecast_2)/2

import numpy as np

def denormalize(forecast):
    max1 = 999.0
    min1 = 0.9
    max2 = 4996.0 
    min2 = 14.37

    forecast = np.asarray(forecast, dtype=float)

    den1 = forecast * (max1 - min1) + min1
    den2 = forecast * (max2 - min2) + min2

    return (den1 + den2) / 2

    

In [ ]:
df = pd.read_csv('E:\FAI_Project\Finance-Management-using-AI\data\prepared_data\Aggregated_Dataset_TS.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
len(df)

In [ ]:
split_index = len(df) - 12
train = df.iloc[0:split_index]
test = df.iloc[split_index:]

In [ ]:
price_df = df['Amount_norm']
price_df.head()

In [ ]:
price_df.plot(figsize=(10,7))
plt.ylabel("Amount_normalized")
plt.title("Months")

In [ ]:
# def denormalize(row):
#     max1 = 4996.0
#     min1 = 14.37
#     if row['origin'] == 'dataset1':
#         return row['Amount_norm'] * (max1 - min1) + min1
#     # else:  # dataset2
#     #     return row['Amount_norm'] * (max2 - min2) + min2

In [ ]:
train_series = train['Amount_norm'].astype(float)
from statsmodels.tsa.api import ExponentialSmoothing

# Assuming 'monthly_data' is your pandas Series with 'Date' as index and 'Price' as values
model = ExponentialSmoothing(train_series,
                             trend='add',
                             seasonal='add',
                             seasonal_periods=12,
                             )

fit = model.fit()
forecast = fit.forecast(steps=12) # Forecast for the next 12 months


ogvalue = forecast * (original_max - original_min) + original_min



In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(test['Amount_norm'], forecast)
print(f"MAE: {mae}")

In [ ]:
denormalized_forecast = denormalize(forecast)

In [ ]:
denormalized_test = denormalize(test['Amount_norm'])

In [ ]:
mae = mean_absolute_error(denormalized_test, denormalized_forecast)
print(f"MAE: {mae}")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Assuming 'monthly_data' is your pandas Series with 'Date' as index and 'Price' as values
# Example parameters - you will need to determine the best ones for your data
# e.g., using auto_arima or ACF/PACF plots
model = SARIMAX(train_series,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12))

fit = model.fit(disp=False)
forecast_2 = fit.forecast(steps=12) # Forecast for the next 12 months


In [ ]:
print(type(forecast_2))
print(forecast_2.head())


In [ ]:
mae = mean_absolute_error(test['Amount_norm'], forecast_2)
print(f"MAE: {mae}")

In [ ]:
denormalized_forecast_2 = denormalize(forecast_2)
# denormalized_test = denormalize(test)

In [ ]:
mae = mean_absolute_error(denormalized_test, denormalized_forecast_2)
print(f"MAE: {mae}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.plot(denormalized_forecast,c='blue')
plt.plot(denormalized_forecast_2, c= 'green')
plt.plot(denormalized_test, c= 'red')
plt.legend()

# Trying Prophet

In [ ]:
# !pip install prophet
# !pip install plotly

In [ ]:
timeseries_df = df[['Date','Amount_norm']]

In [ ]:
# timeseries_df.set_index('Date',inplace= True)
train.head()

In [ ]:
train.rename(columns={'Date':'ds','Amount_norm':'y'},inplace=True)

In [ ]:
timeseries_df = train[['ds','y']]

In [ ]:
p=Prophet(interval_width=0.92,daily_seasonality=False,yearly_seasonality=True)
model= p.fit(timeseries_df)

In [ ]:
future = p.make_future_dataframe(periods=12,freq='M')

In [ ]:
forecast_prediction = p.predict(future)

In [ ]:
forecast_prediction.tail()

In [ ]:
train.tail()

In [ ]:
test

In [ ]:
test

In [ ]:
p.plot(forecast_prediction)
# Customize the plot
plt.scatter(test['Date'],test['Amount_norm'], c='Green')
plt.xlabel('Date')
plt.ylabel('Expense Prediction')
plt.title('Expense Forecast')

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(test['Amount_norm'], forecast_prediction['yhat'][-12:])
print(f"MAE: {mae}")

In [ ]:
denormalized_forecast_3 = denormalize(forecast_prediction['yhat'][-12:])
# denormalized_test = denormalize(test)

In [ ]:
mae = mean_absolute_error(denormalized_test, denormalized_forecast_3)
print(f"MAE: {mae}")

# Trying the model on weekly dataset

In [ ]:
df_key =  pd.read_csv('E:\FAI_Project\Finance-Management-using-AI\data\prepared_data\Forecasting_Master_Dataset.csv')

In [ ]:
df_key.head()

In [ ]:
df_key['Date']

In [ ]:
mask = (df_key['Type'] == 'EXPENSE') | (df_key['Type'] == 'Expense')
filtered = df_key[mask]

# 2) Select columns
TSA = filtered[['Date', 'Normalized_Amount']]
TSA = TSA.sort_values('Date')

In [ ]:
filtered.sort_values('Date')

In [ ]:
split_index = len(filtered) - 30
# train_fitered = TSA.iloc[0:split_index]
test_filtered = filtered.iloc[split_index:]

In [ ]:
test_filtered.count()

In [ ]:
# from sklearn.preprocessing import StandardScaler
# import pandas as pd

# scaler = StandardScaler()

# filtered["Amount"] = (
#     filtered
#     .groupby("dataset")["Amount"]
#     .transform(
#         lambda x: pd.Series(
#             scaler.fit_transform(x.to_frame()).ravel(),
#             index=x.index
#         )
#     )
# )


In [ ]:
filtered

In [ ]:
# # 2) Select columns
# TSA = filtered[['Date', 'Amount']]
# # TSA = TSA.sort_values('Date')
# TSA = TSA.sort_values('Date')

In [ ]:
split_index = len(TSA) - 30
train = TSA.iloc[0:split_index]
test = TSA.iloc[split_index:]

In [ ]:
train.rename(columns={'Date':'ds','Normalized_Amount':'y'},inplace=True)
test.rename(columns={'Date':'ds','Normalized_Amount':'y'},inplace=True)

In [ ]:
p=Prophet(interval_width=0.92,daily_seasonality=False,yearly_seasonality=True)
model= p.fit(train)

In [ ]:
future = p.make_future_dataframe(periods=45,freq='D')

In [ ]:
forecast_prediction = p.predict(future)

In [ ]:
forecast_prediction['ds']

In [ ]:
test

In [ ]:
p.plot(forecast_prediction)
# Customize the plot
plt.scatter(test['ds'],test['y'], c='Green')
plt.xlabel('Date')
plt.ylabel('Expense Prediction')
plt.title('Expense Forecast')

In [ ]:
denormalized_forecast_4 = denormalize(forecast_prediction['yhat'][-45:])
# split_index = len(TSA) - 30
# train = TSA.iloc[0:split_index]
# test = TSA.iloc[split_index:]
denormalized_test = denormalize(test['y'])

In [ ]:
# mae = mean_absolute_error(denormalized_test, denormalized_forecast_4)
# print(f"MAE: {mae}")

In [ ]:
print(forecast_prediction.columns)
print(test.columns)

In [ ]:
# Ensure both have a 'ds' column of dtype datetime64
test["ds"] = pd.to_datetime(test["ds"])
forecast_prediction["ds"] = pd.to_datetime(forecast_prediction["ds"])

merged = test.merge(
    forecast_prediction[["ds", "yhat"]],
    on="ds",
    how="inner"   # only common dates kept
)

print(len(test), len(forecast_prediction), len(merged))  # merged will be <= test


In [ ]:
merged.head()

In [ ]:
merged['Actual_Denormalized'] = test_filtered['Amount'].values

In [ ]:
merged

In [ ]:
mae = mean_absolute_error(merged['y'], merged['yhat'])
print(f"MAE: {mae}")

In [ ]:
denormalized_forecast_4 = denormalize(merged['yhat'])
denormalized_test = denormalize(test['y'])

In [ ]:
mae = mean_absolute_error(denormalized_forecast_4, denormalized_test)
print(f"MAE: {mae}")

In [ ]:
mae = mean_absolute_error(denormalized_forecast_4, merged['Actual_Denormalized'])
print(f"MAE: {mae}")

In [ ]:
from joblib import dump, load
dump(p, "../model/prophet_model.pkl")# after training
# later, in production:

p = load("../model/prophet_model.pkl")


In [ ]:
plt.plot(denormalized_forecast_4, label = ' Predicted')
plt.plot(denormalized_test, label = 'Denormalized')
plt.plot(merged['Actual_Denormalized'], label = 'Actual')
plt.legend()

In [ ]:
fig = p.plot_components(forecast_prediction)
plt.show()

In [ ]:
import seaborn as sns
sns.scatterplot(x='ds', y='y', data=train)
plt.show()

# For brand‑new users (cold start)
With no past data for the user, Prophet alone cannot estimate that user’s personal trend/seasonality. Typical options:​

Use a global or template forecast: e.g., “average user” forecast derived from many users, and show that as a very rough baseline until the user has, say, a few weeks of data.​

Switch to a different modeling setup: a global model (NeuralProphet, panel models, or other sequence models) trained on many users, where user ID or features act as regressors so the model can generalize to new users with little data.​

Only enable personalized Prophet forecasts once a user has accumulated enough days of history (e.g., at least 30–60 days so seasonality/trend can be estimated).

In [ ]:
df_key

# Ensamble method

In [ ]:
import pandas as pd
import numpy as np

def make_sliding_window(df, window=30, horizon=1):
    """
    df: columns ['ds', 'y'], sorted by ds ascending.
    window: number of past days as features.
    horizon: forecast horizon in days (e.g. 1 for next day).
    """
    df = df.sort_values("ds").reset_index(drop=True)
    X, y = [], []
    
    for i in range(window, len(df) - horizon + 1):
        past_vals = df["y"].iloc[i-window:i].values
        target = df["y"].iloc[i + horizon - 1]
        X.append(past_vals)
        y.append(target)
    
    X = np.array(X)
    y = np.array(y)
    
    # Add some calendar features if desired
    ds_used = df["ds"].iloc[window:len(df)-horizon+1].reset_index(drop=True)
    cal = pd.DataFrame({
        "dayofweek": ds_used.dt.dayofweek,
        "month": ds_used.dt.month
    })
    return X, cal, y, ds_used


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

all_preds = {}

# Prepare data
train["ds"] = pd.to_datetime(train["ds"])
# X_lag, X_cal, y, ds_used = make_sliding_window(train, window=60, horizon=30)

X_lag, X_cal, y, ds_used = make_sliding_window(train, window=60, horizon=30)

# Combine lag + calendar features
X = np.hstack([X_lag, X_cal.values])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False  # preserve time order
)

models = {
    "rf": RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ),
    "gbr": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "ridge": Ridge(alpha=1.0)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_val)
    all_preds[name] = preds
    print(name, "MAE:", mean_absolute_error(y_val, preds))

# Simple ensemble: equal weight average
def ensemble_predict(X_input):
    preds = np.column_stack([m.predict(X_input) for m in models.values()])
    return preds.mean(axis=1)


In [ ]:
# Build combined DataFrame
df_plot = pd.DataFrame({"y_val": y_val})
for name, preds in all_preds.items():
    df_plot[name] = preds

# Optional: add dates as index if you have them
# df_plot["date"] = pd.to_datetime(dates_val)
# df_plot = df_plot.set_index("date")

df_last30 = df_plot.iloc[-30:]

plt.figure(figsize=(12, 6))
plt.plot(df_last30.index, df_last30["y_val"], label="y_val", linewidth=2, color="black")
for name in all_preds.keys():
    plt.plot(df_last30.index, df_last30[name], label=name, alpha=0.8)
plt.xlabel("Time step (last 30)")
plt.ylabel("Target")
plt.title("Actual vs model predictions (last 30 points)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

config = {
    "window": 60,
    "horizon": 30
}

joblib.dump(models, "../model/expense_models_1.pkl")
joblib.dump(config, "../model/expense_config_1.pkl")


# Testing

In [ ]:
import numpy as np
import pandas as pd
import joblib

def Load_Model(model_path,config_path):
# Load trained models and config
    models = joblib.load(model_path)   # dict: {"rf": ..., "gbr": ..., "ridge": ...}
    config = joblib.load(config_path)
    WINDOW = config["window"]
    HORIZON = config["horizon"]  # 30
    return models,config,WINDOW,HORIZON

def ensemble_predict_array(X_input):
    preds = np.column_stack([m.predict(X_input) for m in models.values()])
    return preds.mean(axis=1)

def forecast_next_30_days(recent_values, recent_dates,WINDOW,HORIZON):
    """
    recent_values: list/array of last N expenses (N >= WINDOW)
    recent_dates:  list/array of last N dates (strings or datetime), same length as recent_values
    Returns: (forecast_value, future_dates)
        - forecast_value: scalar, predicted average for next 30 days
        - future_dates: pd.DatetimeIndex for those 30 days
    """
    if len(recent_values) < WINDOW:
        raise ValueError(f"Need at least {WINDOW} recent values for forecasting")

    # Take only the most recent WINDOW values
    recent_values = np.array(recent_values[-WINDOW:], dtype=float)
    recent_dates = pd.to_datetime(recent_dates[-WINDOW:])

    # Build lag features: shape (1, WINDOW)
    X_lag = recent_values.reshape(1, -1)

    # Build calendar features using the "prediction anchor" date
    # Here use the last observed date as the anchor
    anchor_date = recent_dates[-1]
    # For the ML model, we use calendar features at anchor time
    X_cal = np.array([[anchor_date.dayofweek, anchor_date.month]], dtype=float)

    # Combine lag + calendar (shape: (1, WINDOW + 2))
    X_input = np.hstack([X_lag, X_cal])

    # Predict average for next 30 days
    avg_30 = ensemble_predict_array(X_input)[0]

    # Build future date index for the next 30 days (if needed for UI)
    future_dates = pd.date_range(start=anchor_date + pd.Timedelta(days=1),
                                 periods=HORIZON, freq="D")

    return avg_30, future_dates


# You have:

y_ds1: values from dataset 1 (low‑range regime).

y_ds2: values from dataset 2 (high‑range regime).

y_sample: recent 30 values for a new user (what you want to classify).

Run a two‑sample K‑S test between y_sample and each dataset:​

ks_1 = ks_2samp(y_sample, y_ds1)

ks_2 = ks_2samp(y_sample, y_ds2)

Compare the K‑S statistic or p‑value:

Smaller statistic / larger p‑value ⇒ distributions are more similar.​

Route to that dataset’s scaler/model.

Example in Python:

In [ ]:
import numpy as np
from scipy.stats import beta, ks_2samp

np.random.seed(42)

def Distributions():
    # Fitted parameters
    a1, b1, loc1, scale1 = 1.53, 17.51, 0.0, 16240.45
    a2, b2, loc2, scale2 = 0.77, 86458.28, 0.0, 10187766.47

    # Reference datasets drawn from the fitted distributions
    data1 = beta(a=a1, b=b1, loc=loc1, scale=scale1).rvs(size=10000)
    data2 = beta(a=a2, b=b2, loc=loc2, scale=scale2).rvs(size=10000)

    return data1,data2

# Your 30-point sample (replace this with your real sample array)
# sample = beta(a=3.0, b=4.0, loc=0, scale=3000).rvs(size=30)
# # 2. Sample you want to classify (beta-distributed)


data1,data2 = Distributions()
sample = beta(a=3.0, b=4.0).rvs(size=60) * 3000

# Two-sample K-S tests
def Hypotesis_testing(sample,data1,data2):
    stat1, p1 = ks_2samp(sample, data1)
    stat2, p2 = ks_2samp(sample, data2)

    print(f"KS vs Dataset 1: stat={stat1:.4f}, p={p1:.4e}")
    print(f"KS vs Dataset 2: stat={stat2:.4f}, p={p2:.4e}")

    # Choose closer distribution by smaller KS statistic (or larger p-value)
    chosen = "Dataset 1" if stat1 < stat2 else "Dataset 2"
    print("Sample is closer to:", chosen)

Hypotesis_testing(sample,data1,data2)


In [ ]:
# max1 = 999.0
# min1 = 0.9
# max2 = 4996.0 
# min2 = 14.37

def normalize_d1(x):
    max1 = 999.0
    min1 = 0.9
    epsilon = 1e-6
    x = np.asarray(x, dtype=float)
    scaled = (x - min1) / (max1 - min1)
    scaled = np.clip(scaled, epsilon, 1 - epsilon)
    return scaled

def denormalize_d1(scaled):
    max1 = 999.0
    min1 = 0.9
    scaled = np.asarray(scaled, dtype=float)
    # assume scaled is already in [epsilon, 1-epsilon]
    x = scaled * (max1 - min1) + min1
    return x

def normalize_d2(x):
    max2 = 4996.0 
    min2 = 14.37
    epsilon = 1e-6
    x = np.asarray(x, dtype=float)
    scaled = (x - min2) / (max2 - min2)
    scaled = np.clip(scaled, epsilon, 1 - epsilon)
    return scaled

def denormalize_d2(scaled):
    max2 = 4996.0 
    min2 = 14.37
    scaled = np.asarray(scaled, dtype=float)
    x = scaled * (max2 - min2) + min2
    return x


In [ ]:
def Choose_Normalization(chosen):
    if chosen == 'Dataset 1':
        return normalize_d1(sample)
    else:
        return normalize_d2(sample)

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. Create 60 synthetic expenses
# -----------------------------
np.random.seed(42)
size = 1200

# Example: mostly low-range spends with some noise
user_expenses = np.random.uniform(200, 1500, size=size).round(2).tolist()

# -----------------------------
# 2. Create 60 dates with gaps
# -----------------------------
start = pd.Timestamp("2025-01-01")
dates = []

current = start
for i in range(size):
    dates.append(current)
    # Random gap: 1–3 days
    gap = np.random.choice([1, 1, 2, 3])   # mostly 1, sometimes 2–3
    current = current + pd.Timedelta(days=gap)

user_dates = [d.strftime("%Y-%m-%d") for d in dates]

print("Number of expenses:", len(user_expenses))
print("Number of dates:", len(user_dates))
print("First 5:")
for e, d in list(zip(user_expenses, user_dates))[:5]:
    print(d, "->", e)

print("Last 5:")
for e, d in list(zip(user_expenses, user_dates))[-5:]:
    print(d, "->", e)


In [ ]:
# -----------------------------
# 3. Call your forecast function
# -----------------------------
# This assumes you have already defined forecast_next_30_days(...)
# which internally: routes distribution -> normalizes -> predicts -> denormalizes

# Two-sample K-S tests
stat1, p1 = ks_2samp(user_expenses, data1)
stat2, p2 = ks_2samp(user_expenses, data2)

print(f"KS vs Dataset 1: stat={stat1:.4f}, p={p1:.4e}")
print(f"KS vs Dataset 2: stat={stat2:.4f}, p={p2:.4e}")

# Choose closer distribution by smaller KS statistic (or larger p-value)
chosen = "Dataset 1" if stat1 < stat2 else "Dataset 2"
print("Sample is closer to:", chosen)

if chosen == 'Dataset 1':
    nomrmalized = normalize_d1(user_expenses)
else:
    nomrmalized = normalize_d2(user_expenses)



# avg_next_30, future_dates = forecast_next_30_days(user_expenses, user_dates)

# if avg_next_30 is None:
#     print("Not enough data yet (cold start).")
# else:
#     print("\nPredicted avg spend next 30 days:", avg_next_30)
#     print("Future dates (next 30 days anchor-based):")
#     print(future_dates)


In [ ]:
models,config,WINDOW,HORIZON = Load_Model()
avg_next_30, future_dates = forecast_next_30_days(nomrmalized, user_dates,WINDOW, HORIZON)

if avg_next_30 is None:
    print("Not enough data yet (cold start).")
else:
    print("\nPredicted avg spend next 30 days:", avg_next_30)
    print("Future dates (next 30 days anchor-based):")
    print(future_dates)

In [ ]:
# avg_next_30

if chosen == 'Dataset 1':
    value = denormalize_d1(avg_next_30)
else:
    value = denormalize_d2(avg_next_30)

In [ ]:
import numpy as np
from scipy import stats

def describe_numeric(x):
    x = np.asarray(x, dtype=float)

    desc = {}
    desc["count"] = x.size
    desc["mean"] = np.mean(x)
    desc["median"] = np.median(x)
    mode_res = stats.mode(x, keepdims=True)
    desc["mode"] = float(mode_res.mode[0]) if mode_res.count[0] > 0 else np.nan

    desc["min"] = np.min(x)
    desc["max"] = np.max(x)
    desc["range"] = desc["max"] - desc["min"]

    desc["var"] = np.var(x, ddof=1)        # sample variance
    desc["std"] = np.std(x, ddof=1)        # sample std

    q25, q50, q75 = np.percentile(x, [25, 50, 75])
    desc["q25"] = q25
    desc["q50"] = q50
    desc["q75"] = q75
    desc["iqr"] = q75 - q25

    desc["skew"] = stats.skew(x, bias=False)
    desc["kurtosis"] = stats.kurtosis(x, bias=False)

    return desc


In [ ]:
describe_numeric(user_expenses)

In [ ]:
value

# New_Model

In [ ]:
import pandas as pd
import numpy as np

def make_sliding_window(df, window=30, horizon=1):
    """
    df: columns ['ds', 'y'], sorted by ds ascending.
    window: number of past days as features.
    horizon: forecast horizon in days (e.g. 1 for next day or avg over next horizon).
    """
    df = df.sort_values("ds").reset_index(drop=True).copy()
    df["ds"] = pd.to_datetime(df["ds"])

    X_lag_list = []
    X_feat_list = []
    y_list = []
    ds_used_list = []

    for i in range(window, len(df) - horizon + 1):
        past_slice = df.iloc[i-window:i]
        target_slice = df.iloc[i:i+horizon]

        # 1) Lag features (raw past values)
        past_vals = past_slice["y"].values  # shape (window,)

        # 2) Rolling stats from past window
        vals = past_vals.astype(float)
        roll_7 = vals[-7:].mean() if len(vals) >= 7 else vals.mean()
        roll_14 = vals[-14:].mean() if len(vals) >= 14 else vals.mean()
        roll_30 = vals[-30:].mean() if len(vals) >= 30 else vals.mean()

        std_7 = vals[-7:].std(ddof=1) if len(vals) >= 7 else vals.std(ddof=1)
        std_14 = vals[-14:].std(ddof=1) if len(vals) >= 14 else vals.std(ddof=1)
        std_30 = vals[-30:].std(ddof=1) if len(vals) >= 30 else vals.std(ddof1)

        # 3) Calendar features based on prediction anchor date
        anchor_ds = df["ds"].iloc[i + horizon - 1]

        dayofweek = anchor_ds.dayofweek
        month = anchor_ds.month
        dayofmonth = anchor_ds.day
        is_weekend = 1 if dayofweek >= 5 else 0
        # pay-day flag: near 1st or 30th (tune as needed)
        is_payday = 1 if dayofmonth in [1, 30, 31] else 0

        # Combine non-lag features into one vector
        extra_feats = [
            dayofweek, month, dayofmonth, is_weekend, is_payday,
            roll_7, roll_14, roll_30,
            std_7, std_14, std_30,
        ]

        # 4) Target: here use mean of next 'horizon' days (30-day avg)
        target = target_slice["y"].mean()

        X_lag_list.append(past_vals)
        X_feat_list.append(extra_feats)
        y_list.append(target)
        ds_used_list.append(anchor_ds)

    X_lag = np.array(X_lag_list)                 # (n_samples, window)
    X_extra = np.array(X_feat_list)              # (n_samples, n_extra_features)
    y = np.array(y_list)                         # (n_samples,)
    ds_used = pd.Series(ds_used_list)

    return X_lag, X_extra, y, ds_used


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

train["ds"] = pd.to_datetime(train["ds"])

# Use window=60 and horizon=30 to predict 30-day avg
X_lag, X_extra, y, ds_used = make_sliding_window(train, window=60, horizon=30)

# Combine lag + extra features
X = np.hstack([X_lag, X_extra])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

models = {
    "rf": RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42
    ),
    "gbr": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "ridge": Ridge(alpha=1.0)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_val)
    print(name, "MAE:", mean_absolute_error(y_val, preds))

def ensemble_predict(X_input):
    preds = np.column_stack([m.predict(X_input) for m in models.values()])
    return preds.mean(axis=1)


In [ ]:
import joblib

config = {
    "window": 60,
    "horizon": 30
}

joblib.dump(models, "../model/expense_models_2.pkl")
joblib.dump(config, "../model/expense_config_2.pkl")


In [ ]:
import numpy as np
import pandas as pd
import joblib

def Load_Model():
# Load trained models and config
    models = joblib.load("../model/expense_models_2.pkl")   # dict: {"rf": ..., "gbr": ..., "ridge": ...}
    config = joblib.load("../model/expense_config_2.pkl")
    WINDOW = config["window"]
    HORIZON = config["horizon"]  # 30
    return models,config,WINDOW,HORIZON

def ensemble_predict_array(X_input):
    preds = np.column_stack([m.predict(X_input) for m in models.values()])
    return preds.mean(axis=1)

def forecast_next_30_days(recent_values, recent_dates, window=60, horizon=30):
    if len(recent_values) < window:
        return None, None

    vals = np.array(recent_values[-window:], dtype=float)
    dates = pd.to_datetime(recent_dates[-window:])

    # lag features (in same scale as training)
    X_lag = vals.reshape(1, -1)

    # rolling stats (use same choice as in training)
    v = vals
    roll_7  = v[-7:].mean()  if len(v) >= 7  else v.mean()
    roll_14 = v[-14:].mean() if len(v) >= 14 else v.mean()
    roll_30 = v[-30:].mean() if len(v) >= 30 else v.mean()

    std_7  = v[-7:].std(ddof=1)  if len(v) >= 7  else v.std(ddof=1)
    std_14 = v[-14:].std(ddof=1) if len(v) >= 14 else v.std(ddof=1)
    std_30 = v[-30:].std(ddof=1) if len(v) >= 30 else v.std(ddof=1)

    anchor_date = dates[-1]
    dayofweek  = anchor_date.dayofweek
    month      = anchor_date.month
    dayofmonth = anchor_date.day
    is_weekend = 1 if dayofweek >= 5 else 0
    is_payday  = 1 if dayofmonth in [1, 30, 31] else 0

    # EXACT same order and count as training
    X_cal = np.array([[dayofweek, month, dayofmonth,
                       is_weekend, is_payday,
                       roll_7, roll_14, roll_30,
                       std_7, std_14, std_30]], dtype=float)

    X_input = np.hstack([X_lag, X_cal])  # shape (1, 60 + 11 = 71)

    avg_30_norm = ensemble_predict(X_input)[0]

    future_dates = pd.date_range(
        start=anchor_date + pd.Timedelta(days=1),
        periods=horizon,
        freq="D"
    )
    return avg_30_norm, future_dates



In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# 1. Create 60 synthetic expenses
# -----------------------------
np.random.seed(42)
size = 1200

# Example: mostly low-range spends with some noise
user_expenses = np.random.uniform(0, 3000, size=size).round(2).tolist()

# -----------------------------
# 2. Create 60 dates with gaps
# -----------------------------
start = pd.Timestamp("2025-01-01")
dates = []

current = start
for i in range(size):
    dates.append(current)
    # Random gap: 1–3 days
    gap = np.random.choice([1, 1, 2, 3])   # mostly 1, sometimes 2–3
    current = current + pd.Timedelta(days=gap)

user_dates = [d.strftime("%Y-%m-%d") for d in dates]


print("Number of expenses:", len(user_expenses))
print("Number of dates:", len(user_dates))
print("First 5:")
for e, d in list(zip(user_expenses, user_dates))[:5]:
    print(d, "->", e)

print("Last 5:")
for e, d in list(zip(user_expenses, user_dates))[-5:]:
    print(d, "->", e)


stat1, p1 = ks_2samp(user_expenses, data1)
stat2, p2 = ks_2samp(user_expenses, data2)

print(f"KS vs Dataset 1: stat={stat1:.4f}, p={p1:.4e}")
print(f"KS vs Dataset 2: stat={stat2:.4f}, p={p2:.4e}")

# Choose closer distribution by smaller KS statistic (or larger p-value)
chosen = "Dataset 1" if stat1 < stat2 else "Dataset 2"
print("Sample is closer to:", chosen)

if chosen == 'Dataset 1':
    normalized_values = normalize_d1(user_expenses)
else:
    normalized_values = normalize_d2(user_expenses)
    



In [ ]:
models,config,WINDOW,HORIZON = Load_Model()
avg_next_30, future_dates = forecast_next_30_days(normalized_values, user_dates,WINDOW, HORIZON)

if avg_next_30 is None:
    print("Not enough data yet (cold start).")
else:
    print("\nPredicted avg spend next 30 days:", avg_next_30)
    print("Future dates (next 30 days anchor-based):")
    print(future_dates)

In [ ]:
if chosen == 'Dataset 1':
        avg_next_30 = denormalize_d1(avg_next_30)
else:
        avg_next_30 = denormalize_d2(avg_next_30)
    # print("\nPredicted avg spend next 30 days:", avg_next_30)
    # print("Future dates (next 30 days anchor-based):")
print(avg_next_30)
print(future_dates)


# Results

In [ ]:
base_mae = {
    "Exponential_Smoothing": 1.9365034105851124,
    "Sarima": 2.0568495592357023,
    "Prophet (no agg)": 0.08421906014583577,
    "RF (baseline)": 0.10839902600596292,
    "GBR (baseline)": 0.10812907952150788,
    "Ridge (baseline)": 0.11089540399253732,
}

tuned_mae = {
    "RF (tuned)": 0.039053318627506765,
    "GBR (tuned)": 0.03961060060812652,
    "Ridge (tuned)": 0.03330891856138044,
}


In [ ]:
import matplotlib.pyplot as plt

# combine all into one dict
all_mae = {
    "ExpSmooth": 1.9365,
    "Sarima": 2.0568,
    "Prophet": 0.0842,
    "RF base": 0.1084,
    "GBR base": 0.1081,
    "Ridge base": 0.1109,
    # "RF tuned": 0.0391,
    # "GBR tuned": 0.0396,
    # "Ridge tuned": 0.0333,
}

models = list(all_mae.keys())
values = list(all_mae.values())

plt.figure(figsize=(10, 4))
plt.bar(models, values, color="skyblue")
plt.ylabel("MAE (lower is better)")
plt.xticks(rotation=45, ha="right")
plt.title("Model MAE Comparison")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = ["RF", "GBR", "Ridge"]
baseline = [0.10839902600596292, 0.10812907952150788, 0.11089540399253732]
tuned    = [0.039053318627506765, 0.03961060060812652, 0.03330891856138044]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
rects1 = ax.bar(x - width/2, baseline, width, label="Baseline", color="lightgray")
rects2 = ax.bar(x + width/2, tuned, width, label="Tuned", color="steelblue")

ax.set_ylabel("MAE (lower is better)")
ax.set_title("Baseline vs Tuned MAE")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
plt.tight_layout()
plt.show()
